# 03 — YouTube Comment Crawling

## Judul Project

**Analisis Sentimen Komentar YouTube terhadap Isu Pelemahan Nilai Rupiah dan Dampaknya terhadap Persepsi Daya Beli Masyarakat Menggunakan Python, Jupyter Notebook, GitHub, dan Streamlit**

## Tujuan Notebook

Notebook ini digunakan untuk menjalankan proses **pencarian video YouTube** dan **crawling komentar YouTube** menggunakan YouTube Data API v3.

Output utama dari notebook ini adalah dataset raw komentar YouTube yang disimpan ke folder:

`data/raw/`

Dataset raw ini akan digunakan pada notebook berikutnya:

`04_data_understanding_raw_audit.ipynb`

## Batasan Notebook

Notebook ini hanya mencakup:

1. Membaca konfigurasi project.
2. Membaca API key dari file `.env`.
3. Memvalidasi koneksi ke YouTube Data API v3.
4. Mencari video berdasarkan keyword.
5. Mengambil metadata video.
6. Mengambil komentar top-level dari setiap video.
7. Menyimpan dataset raw ke folder `data/raw/`.
8. Menyimpan ringkasan crawling yang aman ke folder `reports/`.

Notebook ini **belum melakukan**:

- Cleaning teks.
- Text preprocessing.
- Labeling sentimen.
- Modeling machine learning.
- Evaluasi model.
- Deployment Streamlit.

## Prinsip Keamanan

1. API key tidak ditulis langsung di notebook.
2. API key tidak ditampilkan pada output notebook.
3. Dataset raw masih memuat kolom `author`, sehingga file dataset raw tidak boleh diunggah ke GitHub.
4. Sampel data yang ditampilkan pada notebook menggunakan masking atau penghapusan kolom sensitif.
5. File `.env` dan folder `data/raw/` wajib masuk ke `.gitignore`.

## 1. Import Library

Bagian ini memuat library dasar yang dibutuhkan untuk membaca konfigurasi project, membangun koneksi API, melakukan crawling komentar, membentuk DataFrame, dan menyimpan dataset raw.

In [21]:
# ============================================================
# 1. Import Library
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import os
import json
import time
import yaml

import pandas as pd
import numpy as np

from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

print("Library berhasil di-import.")

Library berhasil di-import.


## 2. Deteksi Project Root dan Folder Utama

Project root adalah folder utama project:

`youtube-rupiah-sentiment-analysis`

Deteksi project root diperlukan agar notebook tetap dapat dijalankan dari folder `notebooks/` maupun dari folder utama project.

Folder utama yang digunakan:

- `data/raw/` untuk menyimpan dataset raw hasil crawling.
- `reports/` untuk menyimpan ringkasan crawling yang aman.

In [22]:
# ============================================================
# 2. Deteksi Project Root dan Folder Utama
# ============================================================

def find_project_root(start_path=None):
    """
    Mendeteksi root folder project secara fleksibel.

    Indikator yang digunakan:
    - config.yaml
    - .git
    - kombinasi folder notebooks dan data
    """
    current_path = Path(start_path or Path.cwd()).resolve()

    for candidate in [current_path] + list(current_path.parents):
        has_config = (candidate / "config.yaml").exists()
        has_git = (candidate / ".git").exists()
        has_notebooks = (candidate / "notebooks").exists()
        has_data = (candidate / "data").exists()

        if has_config or has_git or (has_notebooks and has_data):
            return candidate

    raise FileNotFoundError(
        "Project root tidak ditemukan. Pastikan notebook berada di dalam folder project."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
DATA_RAW_DIR = DATA_DIR / "raw"
REPORTS_DIR = PROJECT_ROOT / "reports"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

# Membuat folder yang dibutuhkan jika belum tersedia
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root berhasil ditemukan:")
print(PROJECT_ROOT)
print("\nFolder data/raw tersedia:", DATA_RAW_DIR.exists())
print("Folder reports tersedia :", REPORTS_DIR.exists())

Project root berhasil ditemukan:
D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis

Folder data/raw tersedia: True
Folder reports tersedia : True


## 3. Validasi File `.env`

API key YouTube Data API v3 dibaca dari file `.env`, bukan ditulis langsung di notebook.

Format minimal file `.env`:

```text
YOUTUBE_API_KEY=isi_api_key_asli
```

Nilai API key tidak akan ditampilkan pada output notebook.

In [23]:
# ============================================================
# 3. Load API Key dari File .env
# ============================================================

ENV_PATH = PROJECT_ROOT / ".env"

if not ENV_PATH.exists():
    raise FileNotFoundError(
        f"File .env tidak ditemukan di: {ENV_PATH}\n"
        "Pastikan file .env berada di root folder project."
    )

load_dotenv(dotenv_path=ENV_PATH)

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

if not YOUTUBE_API_KEY:
    raise ValueError(
        "YOUTUBE_API_KEY tidak ditemukan di file .env. "
        "Pastikan formatnya: YOUTUBE_API_KEY=isi_api_key_asli"
    )

print("File .env berhasil dibaca.")
print("YOUTUBE_API_KEY ditemukan.")
print("API key tidak ditampilkan untuk menjaga keamanan credential.")

File .env berhasil dibaca.
YOUTUBE_API_KEY ditemukan.
API key tidak ditampilkan untuk menjaga keamanan credential.


## 4. Load Konfigurasi dari `config.yaml`

File `config.yaml` digunakan untuk menyimpan parameter crawling, seperti keyword pencarian, jumlah video target, target komentar, region, bahasa relevansi, dan jumlah komentar per video.

Jika sebagian parameter belum tersedia di `config.yaml`, notebook akan menggunakan nilai default yang aman untuk tahap crawling awal.

In [ ]:
# ============================================================
# 4. Load Konfigurasi dari config.yaml
# ============================================================

CONFIG_PATH = PROJECT_ROOT / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"File config.yaml tidak ditemukan di: {CONFIG_PATH}\n"
        "Pastikan file config.yaml berada di root folder project."
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file) or {}

print("File config.yaml berhasil dibaca.")
print("Top-level key yang tersedia:", list(config.keys()))

## 5. Ambil Parameter Crawling

Parameter utama yang digunakan:

- `keywords`: daftar keyword pencarian video.
- `region_code`: region pencarian, default `ID`.
- `relevance_language`: bahasa relevansi, default `id`.
- `order`: urutan pencarian video.
- `search_max_results_per_keyword`: jumlah maksimum video per keyword.
- `max_videos`: jumlah video maksimum yang akan diproses.
- `comments_per_video`: komentar maksimum per video.
- `target_comments_min`: target minimum komentar.
- `target_comments_max`: target maksimum komentar.
- `comment_order`: urutan komentar, misalnya `relevance` atau `time`.

Parameter ini dapat disesuaikan di `config.yaml`.

In [24]:
# ============================================================
# 5. Ambil Parameter Crawling
# ============================================================

youtube_config = config.get("youtube", {})
crawling_config = config.get("crawling", {})

default_keywords = [
    "rupiah melemah daya beli masyarakat",
    "nilai rupiah melemah",
    "dampak rupiah melemah",
    "rupiah melemah harga kebutuhan naik",
    "pelemahan rupiah ekonomi masyarakat"
]

keywords = youtube_config.get("keywords", default_keywords)

# Jika keywords ditulis sebagai string tunggal di YAML, ubah menjadi list
if isinstance(keywords, str):
    keywords = [keywords]

# Buang keyword kosong dan duplikasi sederhana
keywords = list(dict.fromkeys([str(keyword).strip() for keyword in keywords if str(keyword).strip()]))

region_code = youtube_config.get("region_code", "ID")
relevance_language = youtube_config.get("relevance_language", "id")
order = youtube_config.get("order", "relevance")
search_max_results_per_keyword = int(youtube_config.get("search_max_results_per_keyword", 5))

# Parameter opsional rentang tanggal video
published_after = youtube_config.get("published_after", None)
published_before = youtube_config.get("published_before", None)

target_comments_min = int(crawling_config.get("target_comments_min", 6000))
target_comments_max = int(crawling_config.get("target_comments_max", 8000))
max_videos = int(crawling_config.get("max_videos", 20))
comments_per_video = int(crawling_config.get("comments_per_video", 500))
text_format = crawling_config.get("text_format", "plainText")
comment_order = crawling_config.get("comment_order", "relevance")
request_delay_seconds = float(crawling_config.get("request_delay_seconds", 0.2))
sort_videos_by_comments = bool(crawling_config.get("sort_videos_by_comments", True))

# Validasi nilai dasar
if not keywords:
    raise ValueError("Daftar keywords kosong. Tambahkan keyword pada config.yaml.")

if target_comments_max < target_comments_min:
    raise ValueError("target_comments_max tidak boleh lebih kecil dari target_comments_min.")

if comments_per_video <= 0:
    raise ValueError("comments_per_video harus lebih besar dari 0.")

if max_videos <= 0:
    raise ValueError("max_videos harus lebih besar dari 0.")

print("Parameter crawling berhasil dimuat.")
print("Jumlah keyword                 :", len(keywords))
print("Region                         :", region_code)
print("Bahasa relevansi               :", relevance_language)
print("Order pencarian video          :", order)
print("Maksimal hasil per keyword     :", search_max_results_per_keyword)
print("Maksimal video diproses        :", max_videos)
print("Komentar maksimal per video    :", comments_per_video)
print("Target komentar                :", f"{target_comments_min} - {target_comments_max}")
print("Order komentar                 :", comment_order)
print("Request delay seconds          :", request_delay_seconds)
print("Sort video berdasarkan komentar:", sort_videos_by_comments)

Parameter crawling berhasil dimuat.
Jumlah keyword                 : 5
Region                         : ID
Bahasa relevansi               : id
Order pencarian video          : relevance
Maksimal hasil per keyword     : 5
Maksimal video diproses        : 20
Komentar maksimal per video    : 500
Target komentar                : 6000 - 8000
Order komentar                 : relevance
Request delay seconds          : 0.2
Sort video berdasarkan komentar: True


## 6. Membuat Service Object YouTube Data API v3

Service object digunakan untuk melakukan request ke YouTube Data API v3.

API key tetap berada di environment variable dan tidak ditampilkan pada output.

In [25]:
# ============================================================
# 6. Membuat Service Object YouTube Data API v3
# ============================================================

youtube = build(
    serviceName="youtube",
    version="v3",
    developerKey=YOUTUBE_API_KEY
)

print("Service object YouTube Data API v3 berhasil dibuat.")

Service object YouTube Data API v3 berhasil dibuat.


## 7. Validasi Koneksi Awal ke YouTube Data API v3

Validasi koneksi dilakukan dengan request ringan ke channel publik. Tujuannya memastikan:

1. API key terbaca.
2. YouTube Data API v3 sudah aktif.
3. Koneksi API dapat digunakan.
4. Quota API masih tersedia.

Output validasi tidak menampilkan API key.

In [26]:
# ============================================================
# 7. Validasi Koneksi Awal ke YouTube Data API v3
# ============================================================

def parse_http_error(error: HttpError) -> dict:
    """
    Membaca detail HttpError dari Google API secara aman tanpa menampilkan credential.
    """
    error_status = getattr(error.resp, "status", "Unknown")

    try:
        error_content = json.loads(error.content.decode("utf-8"))
        error_reason = error_content.get("error", {}).get("errors", [{}])[0].get("reason", "Unknown reason")
        error_message = error_content.get("error", {}).get("message", "Unknown message")
    except Exception:
        error_reason = "Tidak dapat membaca reason error."
        error_message = "Tidak dapat membaca message error."

    return {
        "http_status": error_status,
        "reason": error_reason,
        "message": error_message
    }


def validate_youtube_api_connection(youtube_service):
    """
    Melakukan validasi koneksi awal ke YouTube Data API v3.
    """
    try:
        request = youtube_service.channels().list(
            part="snippet",
            id="UC_x5XG1OV2P6uZZ5FSM9Ttw",
            maxResults=1
        )

        response = request.execute()

        if not response.get("items"):
            return {
                "status": "warning",
                "message": "Request berhasil, tetapi data channel publik tidak ditemukan."
            }

        channel_title = response["items"][0]["snippet"].get("title", "Unknown Channel")

        return {
            "status": "success",
            "message": "Koneksi ke YouTube Data API v3 berhasil.",
            "sample_channel_title": channel_title
        }

    except HttpError as error:
        error_info = parse_http_error(error)
        return {
            "status": "failed",
            **error_info
        }


validation_result = validate_youtube_api_connection(youtube)

print("Status validasi:", validation_result["status"])
print("Pesan:", validation_result.get("message"))

if validation_result["status"] == "success":
    print("Contoh channel publik terbaca:", validation_result["sample_channel_title"])
else:
    print("HTTP Status:", validation_result.get("http_status"))
    print("Reason:", validation_result.get("reason"))
    print("Catatan: Periksa apakah YouTube Data API v3 sudah enabled, API key benar, dan quota masih tersedia.")

Status validasi: success
Pesan: Koneksi ke YouTube Data API v3 berhasil.
Contoh channel publik terbaca: Google for Developers


## 8. Fungsi Pencarian Video YouTube

Bagian ini membuat fungsi untuk mencari video berdasarkan keyword. Metadata awal yang diambil meliputi:

- `video_id`
- `video_title`
- `channel_id`
- `channel_title`
- `video_published_at`
- `video_url`
- `keyword`

Hasil pencarian video akan dideduplikasi berdasarkan `video_id`.

In [27]:
# ============================================================
# 8. Fungsi Pencarian Video YouTube
# ============================================================

def search_youtube_videos(
    youtube_service,
    keyword,
    max_results=5,
    region_code="ID",
    relevance_language="id",
    order="relevance",
    published_after=None,
    published_before=None
):
    """
    Mencari video YouTube berdasarkan keyword.

    Output:
    - list of dictionary berisi metadata dasar video.
    """
    request_params = {
        "part": "snippet",
        "q": keyword,
        "type": "video",
        "maxResults": min(max_results, 50),
        "regionCode": region_code,
        "relevanceLanguage": relevance_language,
        "order": order
    }

    if published_after:
        request_params["publishedAfter"] = published_after

    if published_before:
        request_params["publishedBefore"] = published_before

    try:
        request = youtube_service.search().list(**request_params)
        response = request.execute()

        video_items = []

        for item in response.get("items", []):
            video_id = item.get("id", {}).get("videoId")
            snippet = item.get("snippet", {})

            if not video_id:
                continue

            video_items.append({
                "keyword": keyword,
                "video_id": video_id,
                "video_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "video_published_at": snippet.get("publishedAt"),
                "video_url": f"https://www.youtube.com/watch?v={video_id}"
            })

        return video_items

    except HttpError as error:
        error_info = parse_http_error(error)
        print(f"Error saat mencari video untuk keyword: {keyword}")
        print("HTTP Status:", error_info.get("http_status"))
        print("Reason:", error_info.get("reason"))
        print("Message:", error_info.get("message"))
        return []

## 9. Menjalankan Pencarian Video

Bagian ini menjalankan pencarian video untuk seluruh keyword yang sudah ditentukan.

Hasilnya disimpan dalam DataFrame `videos_df`.

In [28]:
# ============================================================
# 9. Menjalankan Pencarian Video
# ============================================================

all_video_results = []

print("Mulai pencarian video YouTube...")

for keyword_idx, keyword in enumerate(keywords, start=1):
    print(f"\n[{keyword_idx}/{len(keywords)}] Mencari video untuk keyword: {keyword}")

    video_results = search_youtube_videos(
        youtube_service=youtube,
        keyword=keyword,
        max_results=search_max_results_per_keyword,
        region_code=region_code,
        relevance_language=relevance_language,
        order=order,
        published_after=published_after,
        published_before=published_before
    )

    all_video_results.extend(video_results)

    print("Jumlah video ditemukan:", len(video_results))

    if request_delay_seconds > 0:
        time.sleep(request_delay_seconds)

videos_df = pd.DataFrame(all_video_results)

if videos_df.empty:
    raise ValueError(
        "Tidak ada video yang ditemukan. "
        "Coba ubah keyword, naikkan search_max_results_per_keyword, atau periksa quota API."
    )

# Deduplikasi berdasarkan video_id
videos_df = (
    videos_df
    .drop_duplicates(subset=["video_id"], keep="first")
    .reset_index(drop=True)
)

print("\nPencarian video selesai.")
print("Jumlah video unik ditemukan:", len(videos_df))

display(videos_df.head(10))

Mulai pencarian video YouTube...

[1/5] Mencari video untuk keyword: rupiah melemah daya beli masyarakat
Jumlah video ditemukan: 5

[2/5] Mencari video untuk keyword: nilai rupiah melemah
Jumlah video ditemukan: 5

[3/5] Mencari video untuk keyword: dampak rupiah melemah
Jumlah video ditemukan: 5

[4/5] Mencari video untuk keyword: rupiah melemah harga kebutuhan naik
Jumlah video ditemukan: 5

[5/5] Mencari video untuk keyword: pelemahan rupiah ekonomi masyarakat
Jumlah video ditemukan: 5

Pencarian video selesai.
Jumlah video unik ditemukan: 18


,keyword,video_id,video_title,channel_id,channel_title,video_published_at,video_url
0,rupiah melemah daya beli masyarakat,--_k_ACPYTk,"Rupiah Makin Lemah, Daya Beli Mulai Goyah? | Prime Plus",UCKII0Ml9S5wneKbHswmUrIQ,CNN Indonesia,2026-05-25T15:08:01Z,https://www.youtube.com/watch?v=--_k_ACPYTk
1,rupiah melemah daya beli masyarakat,GXLA2I3bifE,"Rupiah Melemah Tajam Rp17 800, Menkeu Purbaya: Tak Masuk Akal, Fundamental Ekonomi Kita Bagus!",UCYNpIdSp5xvvQzgOHtw2RAw,KOMPASTV JAWA TIMUR,2026-05-27T13:20:37Z,https://www.youtube.com/watch?v=GXLA2I3bifE
2,rupiah melemah daya beli masyarakat,Wcq5TnmXQXU,"Rupiah Tembus Level Terlemah, Dampak Merembet ke Harga Kebutuhan Pokok | BEDAH DATA",UCAMpZJJNQPZ6q7ZYKJV1igQ,Nusantara TV,2026-05-27T06:00:39Z,https://www.youtube.com/watch?v=Wcq5TnmXQXU
3,rupiah melemah daya beli masyarakat,vvLBbx0q9F4,Roy Mandey: Masyarakat Cenderung Tidak Lagi Belanja Bulanan | Prime Plus Part 1,UCKII0Ml9S5wneKbHswmUrIQ,CNN Indonesia,2026-05-26T00:30:19Z,https://www.youtube.com/watch?v=vvLBbx0q9F4
4,rupiah melemah daya beli masyarakat,QkD-iloEgo0,Rupiah Bisa Tembus Rp17.000? Ini Dampaknya,UCK_hyMyoOcqDso0vhPKtmNw,DailyNewsIndonesia,2026-05-27T08:53:25Z,https://www.youtube.com/watch?v=QkD-iloEgo0
5,nilai rupiah melemah,sGgYsNt2ick,Akar Masalah Pelemahan Rupiah,UCOcta1Ij35IrXMIbhed3MNg,Ngomongin Uang,2026-05-24T07:00:25Z,https://www.youtube.com/watch?v=sGgYsNt2ick
6,nilai rupiah melemah,0BrRGQjIvwE,"Rupiah Melemah, Pakar Soroti Urgensi Kunjungan Kerja ke Luar Negeri",UCPAxpUn1mrn14xU0JpsLhDg,Kompas.com,2026-05-29T00:04:38Z,https://www.youtube.com/watch?v=0BrRGQjIvwE
7,nilai rupiah melemah,LHPZRgUL6tg,"Rupiah Melemah, Apa yang Terjadi jika Tembus Rp 25.000 Per Dollar AS?",UCPAxpUn1mrn14xU0JpsLhDg,Kompas.com,2026-05-25T07:58:23Z,https://www.youtube.com/watch?v=LHPZRgUL6tg
8,nilai rupiah melemah,nF0McI5sqBA,Sejarah Rupiah: Kenapa Nilainya Selalu Turun? KATA BUZZER STRATEGI PEMERINTAH! |Learning By Googling,UCfQHaUbD0oEBH_FRYHE5qIg,Sepulang Sekolah,2026-05-27T10:00:29Z,https://www.youtube.com/watch?v=nF0McI5sqBA
9,dampak rupiah melemah,xrzhniNhZEs,"[FULL] Dampak Rupiah Melemah, KOWANTARA: Harga Bahan Baku Paling Sensitif Bagi Pengusaha Warteg",UCAMpZJJNQPZ6q7ZYKJV1igQ,Nusantara TV,2026-05-27T05:45:32Z,https://www.youtube.com/watch?v=xrzhniNhZEs


## 10. Enrichment Metadata Video

Bagian ini mengambil metadata tambahan dari setiap video, terutama statistik seperti:

- `view_count`
- `like_count_video`
- `comment_count`

Metadata ini berguna untuk memilih video yang memiliki potensi komentar lebih banyak.

In [29]:
# ============================================================
# 10. Enrichment Metadata Video
# ============================================================

def enrich_video_statistics(youtube_service, video_ids):
    """
    Mengambil statistik video berdasarkan daftar video_id.
    YouTube API videos().list menerima maksimal 50 video_id per request.
    """
    enriched_rows = []

    for start_idx in range(0, len(video_ids), 50):
        batch_ids = video_ids[start_idx:start_idx + 50]

        try:
            request = youtube_service.videos().list(
                part="snippet,statistics",
                id=",".join(batch_ids)
            )
            response = request.execute()

            for item in response.get("items", []):
                snippet = item.get("snippet", {})
                statistics = item.get("statistics", {})

                enriched_rows.append({
                    "video_id": item.get("id"),
                    "video_title_api": snippet.get("title"),
                    "channel_title_api": snippet.get("channelTitle"),
                    "video_published_at_api": snippet.get("publishedAt"),
                    "view_count": statistics.get("viewCount"),
                    "like_count_video": statistics.get("likeCount"),
                    "comment_count": statistics.get("commentCount")
                })

        except HttpError as error:
            error_info = parse_http_error(error)
            print("Error saat mengambil metadata video.")
            print("HTTP Status:", error_info.get("http_status"))
            print("Reason:", error_info.get("reason"))
            print("Message:", error_info.get("message"))

        if request_delay_seconds > 0:
            time.sleep(request_delay_seconds)

    return pd.DataFrame(enriched_rows)


video_stats_df = enrich_video_statistics(youtube, videos_df["video_id"].tolist())

if video_stats_df.empty:
    print("Metadata statistik video tidak berhasil diambil. Proses tetap dapat dilanjutkan.")
    videos_enriched_df = videos_df.copy()
    videos_enriched_df["comment_count"] = np.nan
else:
    videos_enriched_df = videos_df.merge(video_stats_df, on="video_id", how="left")

    # Gunakan hasil API terbaru jika tersedia
    videos_enriched_df["video_title"] = videos_enriched_df["video_title_api"].fillna(videos_enriched_df["video_title"])
    videos_enriched_df["channel_title"] = videos_enriched_df["channel_title_api"].fillna(videos_enriched_df["channel_title"])
    videos_enriched_df["video_published_at"] = videos_enriched_df["video_published_at_api"].fillna(videos_enriched_df["video_published_at"])

    videos_enriched_df = videos_enriched_df.drop(
        columns=["video_title_api", "channel_title_api", "video_published_at_api"],
        errors="ignore"
    )

# Konversi statistik numerik
for col in ["view_count", "like_count_video", "comment_count"]:
    if col in videos_enriched_df.columns:
        videos_enriched_df[col] = pd.to_numeric(videos_enriched_df[col], errors="coerce")

# Urutkan berdasarkan jumlah komentar jika diaktifkan
if sort_videos_by_comments and "comment_count" in videos_enriched_df.columns:
    videos_enriched_df = (
        videos_enriched_df
        .sort_values(by="comment_count", ascending=False, na_position="last")
        .reset_index(drop=True)
    )

print("Metadata video berhasil disiapkan.")
print("Jumlah video siap diproses:", len(videos_enriched_df))

display(videos_enriched_df.head(10))

Metadata video berhasil disiapkan.
Jumlah video siap diproses: 18


,keyword,video_id,video_title,channel_id,channel_title,video_published_at,video_url,view_count,like_count_video,comment_count
0,dampak rupiah melemah,sB7fSPbYFJg,Kenapa Rupiah Melemah? (Explained),UCu0yQD7NFMyLu_-TmKa4Hqg,Kok Bisa?,2015-08-23T13:44:50Z,https://www.youtube.com/watch?v=sB7fSPbYFJg,1989064,45181,5403
1,nilai rupiah melemah,nF0McI5sqBA,Sejarah Rupiah: Kenapa Nilainya Selalu Turun? KATA BUZZER STRATEGI PEMERINTAH! |Learning By Googling,UCfQHaUbD0oEBH_FRYHE5qIg,Sepulang Sekolah,2026-05-27T10:00:29Z,https://www.youtube.com/watch?v=nF0McI5sqBA,304370,8014,2059
2,rupiah melemah harga kebutuhan naik,u72ts1RA2oQ,PENJELASAN: Kenapa Harga Dolar Naik & Rupiah Melemah?,UCOcta1Ij35IrXMIbhed3MNg,Ngomongin Uang,2020-03-23T09:45:07Z,https://www.youtube.com/watch?v=u72ts1RA2oQ,397512,11542,1645
3,nilai rupiah melemah,sGgYsNt2ick,Akar Masalah Pelemahan Rupiah,UCOcta1Ij35IrXMIbhed3MNg,Ngomongin Uang,2026-05-24T07:00:25Z,https://www.youtube.com/watch?v=sGgYsNt2ick,137322,3111,1214
4,rupiah melemah daya beli masyarakat,--_k_ACPYTk,"Rupiah Makin Lemah, Daya Beli Mulai Goyah? | Prime Plus",UCKII0Ml9S5wneKbHswmUrIQ,CNN Indonesia,2026-05-25T15:08:01Z,https://www.youtube.com/watch?v=--_k_ACPYTk,90461,712,766
5,dampak rupiah melemah,DVJZCxeBfMA,DAMPAK RUPIAH MELEMAH 1 JADI GILA ❌ SEMUA WARGA JADI GILA ✅🫵🏼😭,UCND0hMRi9tKz8oxqLqXyuog,poin penting,2026-05-24T23:15:12Z,https://www.youtube.com/watch?v=DVJZCxeBfMA,2679908,83054,705
6,rupiah melemah daya beli masyarakat,GXLA2I3bifE,"Rupiah Melemah Tajam Rp17 800, Menkeu Purbaya: Tak Masuk Akal, Fundamental Ekonomi Kita Bagus!",UCYNpIdSp5xvvQzgOHtw2RAw,KOMPASTV JAWA TIMUR,2026-05-27T13:20:37Z,https://www.youtube.com/watch?v=GXLA2I3bifE,46068,177,344
7,nilai rupiah melemah,LHPZRgUL6tg,"Rupiah Melemah, Apa yang Terjadi jika Tembus Rp 25.000 Per Dollar AS?",UCPAxpUn1mrn14xU0JpsLhDg,Kompas.com,2026-05-25T07:58:23Z,https://www.youtube.com/watch?v=LHPZRgUL6tg,28582,224,326
8,rupiah melemah daya beli masyarakat,Wcq5TnmXQXU,"Rupiah Tembus Level Terlemah, Dampak Merembet ke Harga Kebutuhan Pokok | BEDAH DATA",UCAMpZJJNQPZ6q7ZYKJV1igQ,Nusantara TV,2026-05-27T06:00:39Z,https://www.youtube.com/watch?v=Wcq5TnmXQXU,13936,104,152
9,rupiah melemah daya beli masyarakat,vvLBbx0q9F4,Roy Mandey: Masyarakat Cenderung Tidak Lagi Belanja Bulanan | Prime Plus Part 1,UCKII0Ml9S5wneKbHswmUrIQ,CNN Indonesia,2026-05-26T00:30:19Z,https://www.youtube.com/watch?v=vvLBbx0q9F4,24332,143,118


## 11. Menyimpan Daftar Kandidat Video

Daftar kandidat video disimpan ke folder `data/raw/` sebagai dokumentasi awal proses crawling.

File ini berisi metadata video publik dan digunakan untuk menelusuri sumber komentar pada dataset raw.

In [30]:
# ============================================================
# 11. Simpan Daftar Kandidat Video
# ============================================================

crawl_run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

video_candidates_file = DATA_RAW_DIR / f"youtube_video_candidates_{crawl_run_timestamp}.csv"

videos_enriched_df.to_csv(video_candidates_file, index=False, encoding="utf-8-sig")

print("Daftar kandidat video berhasil disimpan.")
print("Lokasi file:", video_candidates_file)

Daftar kandidat video berhasil disimpan.
Lokasi file: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\data\raw\youtube_video_candidates_20260529_141537.csv


## 12. Fungsi Crawling Komentar YouTube

Fungsi berikut mengambil komentar **top-level** dari setiap video.

Kolom utama yang dikumpulkan:

- `video_id`
- `video_title`
- `video_url`
- `channel_title`
- `video_published_at`
- `comment_id`
- `author`
- `published_at`
- `comment_updated_at`
- `text_original`
- `like_count`
- `reply_count`
- `crawl_timestamp`

Catatan:

- Kolom `author` tetap disimpan di dataset raw untuk audit internal, tetapi tidak boleh dipublikasikan ke GitHub.
- Pada preview output, kolom `author` akan dimasking.

In [31]:
# ============================================================
# 12. Fungsi Crawling Komentar YouTube
# ============================================================

def get_video_comments(
    youtube_service,
    video_id,
    video_title=None,
    video_url=None,
    channel_title=None,
    video_published_at=None,
    max_comments=500,
    text_format="plainText",
    comment_order="relevance"
):
    """
    Mengambil komentar top-level YouTube berdasarkan video_id.

    Output:
    - list of dictionary berisi data komentar.
    """
    comments = []
    next_page_token = None

    while len(comments) < max_comments:
        remaining = max_comments - len(comments)

        try:
            request = youtube_service.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(100, remaining),
                pageToken=next_page_token,
                textFormat=text_format,
                order=comment_order
            )

            response = request.execute()

            for item in response.get("items", []):
                thread_snippet = item.get("snippet", {})
                top_level_comment = thread_snippet.get("topLevelComment", {})
                top_comment_snippet = top_level_comment.get("snippet", {})

                comment_data = {
                    "video_id": video_id,
                    "video_title": video_title,
                    "video_url": video_url,
                    "channel_title": channel_title,
                    "video_published_at": video_published_at,
                    "comment_id": top_level_comment.get("id"),
                    "author": top_comment_snippet.get("authorDisplayName"),
                    "published_at": top_comment_snippet.get("publishedAt"),
                    "comment_updated_at": top_comment_snippet.get("updatedAt"),
                    "text_original": top_comment_snippet.get("textDisplay"),
                    "like_count": top_comment_snippet.get("likeCount", 0),
                    "reply_count": thread_snippet.get("totalReplyCount", 0),
                    "crawl_timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
                }

                comments.append(comment_data)

                if len(comments) >= max_comments:
                    break

            next_page_token = response.get("nextPageToken")

            if not next_page_token:
                break

        except HttpError as error:
            error_info = parse_http_error(error)
            print(f"HTTP Error saat crawling video_id={video_id}")
            print("Reason :", error_info.get("reason"))
            print("Message:", error_info.get("message"))
            break

        except Exception as error:
            print(f"Error umum saat crawling video_id={video_id}: {error}")
            break

        if request_delay_seconds > 0:
            time.sleep(request_delay_seconds)

    return comments

## 13. Menjalankan Crawling Komentar

Bagian ini menjalankan crawling komentar dari video yang sudah ditemukan.

Strategi crawling:

1. Ambil maksimal `max_videos` video.
2. Untuk setiap video, ambil maksimal `comments_per_video` komentar.
3. Hentikan proses jika jumlah komentar mencapai `target_comments_max`.
4. Simpan hasil ke DataFrame `comments_df`.

Jika jumlah komentar belum mencapai `target_comments_min`, notebook akan memberikan warning, tetapi dataset tetap dapat disimpan untuk audit awal.

In [32]:
# ============================================================
# 13. Menjalankan Crawling Komentar
# ============================================================

selected_videos_df = videos_enriched_df.head(max_videos).copy().reset_index(drop=True)

all_comments = []

print("Mulai crawling komentar YouTube...")
print("Jumlah video yang akan diproses:", len(selected_videos_df))
print("Target komentar per video      :", comments_per_video)
print("Target total komentar          :", f"{target_comments_min} - {target_comments_max}")

for idx, row in selected_videos_df.iterrows():
    video_number = idx + 1
    video_id = row.get("video_id")
    video_title = row.get("video_title")
    video_url = row.get("video_url")
    channel_title = row.get("channel_title")
    video_published_at = row.get("video_published_at")

    if not video_id:
        print(f"\n[{video_number}/{len(selected_videos_df)}] Video dilewati karena video_id kosong.")
        continue

    print(f"\n[{video_number}/{len(selected_videos_df)}] Memproses video:")
    print("Video ID:", video_id)
    print("Channel :", channel_title)
    print("Judul   :", str(video_title)[:120])

    comments = get_video_comments(
        youtube_service=youtube,
        video_id=video_id,
        video_title=video_title,
        video_url=video_url,
        channel_title=channel_title,
        video_published_at=video_published_at,
        max_comments=comments_per_video,
        text_format=text_format,
        comment_order=comment_order
    )

    all_comments.extend(comments)

    print("Komentar terkumpul dari video ini:", len(comments))
    print("Total komentar sementara         :", len(all_comments))

    if len(all_comments) >= target_comments_max:
        print("\nTarget maksimum komentar tercapai. Crawling dihentikan.")
        break

comments_df = pd.DataFrame(all_comments)

print("\nCrawling komentar selesai.")
print("Jumlah baris comments_df :", comments_df.shape[0])
print("Jumlah kolom comments_df :", comments_df.shape[1])

if comments_df.empty:
    raise ValueError(
        "comments_df kosong. Kemungkinan komentar video dinonaktifkan, quota habis, "
        "atau video yang ditemukan tidak memiliki komentar publik."
    )

if len(comments_df) < target_comments_min:
    print(
        f"WARNING: Jumlah komentar ({len(comments_df)}) masih di bawah target minimum "
        f"({target_comments_min}). Dataset tetap dapat diaudit, tetapi pertimbangkan "
        "menambah keyword, max_videos, atau comments_per_video."
    )
else:
    print("Jumlah komentar sudah memenuhi target minimum.")

Mulai crawling komentar YouTube...
Jumlah video yang akan diproses: 18
Target komentar per video      : 500
Target total komentar          : 6000 - 8000

[1/18] Memproses video:
Video ID: sB7fSPbYFJg
Channel : Kok Bisa?
Judul   : Kenapa Rupiah Melemah? (Explained)
Komentar terkumpul dari video ini: 500
Total komentar sementara         : 500

[2/18] Memproses video:
Video ID: nF0McI5sqBA
Channel : Sepulang Sekolah
Judul   : Sejarah Rupiah: Kenapa Nilainya Selalu Turun? KATA BUZZER STRATEGI PEMERINTAH! |Learning By Googling
Komentar terkumpul dari video ini: 500
Total komentar sementara         : 1000

[3/18] Memproses video:
Video ID: u72ts1RA2oQ
Channel : Ngomongin Uang
Judul   : PENJELASAN: Kenapa Harga Dolar Naik & Rupiah Melemah?
Komentar terkumpul dari video ini: 500
Total komentar sementara         : 1500

[4/18] Memproses video:
Video ID: sGgYsNt2ick
Channel : Ngomongin Uang
Judul   : Akar Masalah Pelemahan Rupiah
Komentar terkumpul dari video ini: 500
Total komentar sementara   

## 14. Validasi Struktur Dataset Raw

Bagian ini memeriksa apakah dataset hasil crawling memiliki kolom minimal yang dibutuhkan untuk tahap audit berikutnya.

Kolom minimal:

- `video_id`
- `video_title`
- `comment_id`
- `author`
- `published_at`
- `text_original`
- `like_count`

In [33]:
# ============================================================
# 14. Validasi Struktur Dataset Raw
# ============================================================

required_columns = [
    "video_id",
    "video_title",
    "comment_id",
    "author",
    "published_at",
    "text_original",
    "like_count"
]

optional_columns = [
    "video_url",
    "channel_title",
    "video_published_at",
    "comment_updated_at",
    "reply_count",
    "crawl_timestamp"
]

missing_required_columns = [
    col for col in required_columns
    if col not in comments_df.columns
]

if missing_required_columns:
    raise ValueError(
        f"Kolom wajib belum lengkap: {missing_required_columns}"
    )

print("Validasi struktur dataset berhasil.")
print("Kolom wajib tersedia semua.")
print("Jumlah baris:", comments_df.shape[0])
print("Jumlah kolom:", comments_df.shape[1])
print("\nDaftar kolom:")
print(list(comments_df.columns))

Validasi struktur dataset berhasil.
Kolom wajib tersedia semua.
Jumlah baris: 3742
Jumlah kolom: 13

Daftar kolom:
['video_id', 'video_title', 'video_url', 'channel_title', 'video_published_at', 'comment_id', 'author', 'published_at', 'comment_updated_at', 'text_original', 'like_count', 'reply_count', 'crawl_timestamp']


## 15. Preview Aman Dataset Raw

Preview data hanya digunakan untuk validasi visual.

Kolom `author` dimasking agar identitas penulis komentar tidak tampil secara terbuka.

In [34]:
# ============================================================
# 15. Preview Aman Dataset Raw
# ============================================================

def mask_author(author):
    """
    Melakukan masking sederhana pada nama author.
    """
    if pd.isna(author):
        return None

    author = str(author)

    if len(author) <= 2:
        return "***"

    return author[:1] + "***" + author[-1:]


safe_preview_df = comments_df.head(5).copy()

if "author" in safe_preview_df.columns:
    safe_preview_df["author"] = safe_preview_df["author"].apply(mask_author)

preview_columns = [
    col for col in [
        "video_id",
        "video_title",
        "channel_title",
        "comment_id",
        "author",
        "published_at",
        "text_original",
        "like_count",
        "reply_count"
    ]
    if col in safe_preview_df.columns
]

display(safe_preview_df[preview_columns])

,video_id,video_title,channel_title,comment_id,author,published_at,text_original,like_count,reply_count
0,sB7fSPbYFJg,Kenapa Rupiah Melemah? (Explained),Kok Bisa?,Ugzo_OAorVrtNPFL3cF4AaABAg,@***y,2026-05-24T01:37:15Z,"gue kira di upload 10 jam yg lalu,ternyata 10 tahun yg lalu😂",560,11
1,sB7fSPbYFJg,Kenapa Rupiah Melemah? (Explained),Kok Bisa?,Ugx2WJ9HFPUNRPaOxON4AaABAg,@***l,2026-05-22T13:31:00Z,"Timing nya gabisa lebih pas dari ini😭🙏\nNyentuh Rp.25.000,00 Masih Aman Indonesia😎👌",570,12
2,sB7fSPbYFJg,Kenapa Rupiah Melemah? (Explained),Kok Bisa?,UgwyjT6JiPsbjsMoUqR4AaABAg,@***i,2026-05-24T09:42:48Z,Algoritma YouTube cukup mengerikan 🥀,271,3
3,sB7fSPbYFJg,Kenapa Rupiah Melemah? (Explained),Kok Bisa?,UgwN7TGK6eU5papwnNZ4AaABAg,@***8,2026-05-21T06:57:48Z,dyeem 10 year ago lewat beranda kuhh pas bgett;(((,321,2
4,sB7fSPbYFJg,Kenapa Rupiah Melemah? (Explained),Kok Bisa?,UgzSaWu-A-vsRbjSc6B4AaABAg,@***r,2026-05-23T17:47:36Z,Algoritma YouTube lebih mengerikan daripada Pria Solo☠️,92,1


## 16. Ringkasan Cepat Hasil Crawling

Bagian ini membuat ringkasan cepat hasil crawling sebelum dataset disimpan.

Ringkasan ini bersifat agregat dan aman untuk dijadikan dokumentasi.

In [35]:
# ============================================================
# 16. Ringkasan Cepat Hasil Crawling
# ============================================================

comments_df["published_at"] = pd.to_datetime(comments_df["published_at"], errors="coerce")
comments_df["like_count"] = pd.to_numeric(comments_df["like_count"], errors="coerce").fillna(0).astype(int)

if "reply_count" in comments_df.columns:
    comments_df["reply_count"] = pd.to_numeric(comments_df["reply_count"], errors="coerce").fillna(0).astype(int)
else:
    comments_df["reply_count"] = 0

crawl_summary = {
    "crawl_run_timestamp": crawl_run_timestamp,
    "total_comments": int(len(comments_df)),
    "total_columns": int(comments_df.shape[1]),
    "total_unique_videos": int(comments_df["video_id"].nunique()),
    "total_unique_channels": int(comments_df["channel_title"].nunique()) if "channel_title" in comments_df.columns else None,
    "min_comment_date": str(comments_df["published_at"].min()) if comments_df["published_at"].notna().any() else None,
    "max_comment_date": str(comments_df["published_at"].max()) if comments_df["published_at"].notna().any() else None,
    "total_like_count": int(comments_df["like_count"].sum()),
    "total_reply_count": int(comments_df["reply_count"].sum()),
    "target_comments_min": int(target_comments_min),
    "target_comments_max": int(target_comments_max),
    "target_status": "memenuhi_target_minimum" if len(comments_df) >= target_comments_min else "belum_memenuhi_target_minimum"
}

crawl_summary_df = pd.DataFrame([crawl_summary])

display(crawl_summary_df)

,crawl_run_timestamp,total_comments,total_columns,total_unique_videos,total_unique_channels,min_comment_date,max_comment_date,total_like_count,total_reply_count,target_comments_min,target_comments_max,target_status
0,20260529_141537,3742,13,17,12,2015-08-23 14:36:01+00:00,2026-05-29 07:14:24+00:00,27764,2827,6000,8000,belum_memenuhi_target_minimum


## 17. Menyimpan Dataset Raw Komentar YouTube

Dataset raw disimpan ke:

`data/raw/`

Nama file menggunakan timestamp agar setiap proses crawling terdokumentasi.

Catatan penting:

- File raw ini masih memuat kolom `author`.
- File raw tidak boleh diunggah ke GitHub.
- Pastikan `.gitignore` memuat `data/raw/`.

In [36]:
# ============================================================
# 17. Simpan Dataset Raw Komentar YouTube
# ============================================================

raw_comments_file = DATA_RAW_DIR / f"youtube_comments_raw_{crawl_run_timestamp}.csv"

comments_df.to_csv(raw_comments_file, index=False, encoding="utf-8-sig")

print("Dataset raw komentar berhasil disimpan.")
print("Jumlah baris :", comments_df.shape[0])
print("Jumlah kolom :", comments_df.shape[1])
print("Lokasi file  :", raw_comments_file)

Dataset raw komentar berhasil disimpan.
Jumlah baris : 3742
Jumlah kolom : 13
Lokasi file  : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\data\raw\youtube_comments_raw_20260529_141537.csv


## 18. Menyimpan Ringkasan Crawling yang Aman

Ringkasan crawling disimpan ke folder `reports/`.

File ini aman karena hanya berisi statistik agregat dan tidak memuat data komentar mentah maupun author.

In [37]:
# ============================================================
# 18. Simpan Ringkasan Crawling Aman
# ============================================================

summary_csv_file = REPORTS_DIR / f"youtube_crawling_summary_{crawl_run_timestamp}.csv"
summary_json_file = REPORTS_DIR / f"youtube_crawling_summary_{crawl_run_timestamp}.json"

crawl_summary_df.to_csv(summary_csv_file, index=False, encoding="utf-8-sig")

with open(summary_json_file, "w", encoding="utf-8") as file:
    json.dump(crawl_summary, file, ensure_ascii=False, indent=4)

print("Ringkasan crawling berhasil disimpan.")
print("CSV :", summary_csv_file)
print("JSON:", summary_json_file)

Ringkasan crawling berhasil disimpan.
CSV : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\reports\youtube_crawling_summary_20260529_141537.csv
JSON: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\reports\youtube_crawling_summary_20260529_141537.json


## 19. Validasi File Output

Bagian ini memastikan file dataset raw dan ringkasan crawling benar-benar tersimpan.

In [38]:
# ============================================================
# 19. Validasi File Output
# ============================================================

output_validation = {
    "raw_comments_file_exists": raw_comments_file.exists(),
    "raw_comments_file_name": raw_comments_file.name,
    "raw_comments_file_size_mb": round(raw_comments_file.stat().st_size / (1024 * 1024), 4) if raw_comments_file.exists() else None,
    "video_candidates_file_exists": video_candidates_file.exists(),
    "video_candidates_file_name": video_candidates_file.name,
    "summary_csv_exists": summary_csv_file.exists(),
    "summary_json_exists": summary_json_file.exists()
}

output_validation_df = pd.DataFrame([output_validation])

display(output_validation_df)

if not raw_comments_file.exists():
    raise FileNotFoundError("File dataset raw komentar tidak ditemukan setelah proses penyimpanan.")

print("Validasi file output selesai.")

,raw_comments_file_exists,raw_comments_file_name,raw_comments_file_size_mb,video_candidates_file_exists,video_candidates_file_name,summary_csv_exists,summary_json_exists
0,True,youtube_comments_raw_20260529_141537.csv,1.3799,True,youtube_video_candidates_20260529_141537.csv,True,True


Validasi file output selesai.


## 20. Checklist Akhir Notebook 03

Setelah notebook ini selesai dijalankan, pastikan kondisi berikut terpenuhi:

1. Service YouTube Data API v3 berhasil dibuat.
2. Validasi koneksi API berhasil.
3. Video kandidat berhasil ditemukan.
4. Komentar berhasil dikumpulkan.
5. `comments_df` memiliki baris data.
6. File dataset raw tersimpan di `data/raw/`.
7. Ringkasan crawling tersimpan di `reports/`.
8. Output notebook tidak menampilkan API key.
9. Folder `data/raw/` dan file `.env` tidak masuk GitHub.

Setelah checklist terpenuhi, lanjutkan ke notebook:

`04_data_understanding_raw_audit.ipynb`

In [39]:
# ============================================================
# 20. Checklist Akhir
# ============================================================

final_checklist = pd.DataFrame([
    {"item": "Service YouTube API tersedia", "status": "OK" if "youtube" in globals() else "CHECK"},
    {"item": "Validasi koneksi API sukses", "status": "OK" if validation_result.get("status") == "success" else "CHECK"},
    {"item": "Video kandidat ditemukan", "status": "OK" if len(videos_enriched_df) > 0 else "CHECK"},
    {"item": "Komentar berhasil dikumpulkan", "status": "OK" if len(comments_df) > 0 else "CHECK"},
    {"item": "Kolom wajib tersedia", "status": "OK" if not missing_required_columns else "CHECK"},
    {"item": "Dataset raw tersimpan", "status": "OK" if raw_comments_file.exists() else "CHECK"},
    {"item": "Ringkasan CSV tersimpan", "status": "OK" if summary_csv_file.exists() else "CHECK"},
    {"item": "Ringkasan JSON tersimpan", "status": "OK" if summary_json_file.exists() else "CHECK"},
    {"item": "Target minimum komentar", "status": "OK" if len(comments_df) >= target_comments_min else "WARNING"}
])

display(final_checklist)

print("Notebook 03 selesai dijalankan.")
print("Silakan lanjut ke notebook 04 untuk audit dataset raw.")

,item,status
0,Service YouTube API tersedia,OK
1,Validasi koneksi API sukses,OK
2,Video kandidat ditemukan,OK
3,Komentar berhasil dikumpulkan,OK
4,Kolom wajib tersedia,OK
5,Dataset raw tersimpan,OK
6,Ringkasan CSV tersimpan,OK
7,Ringkasan JSON tersimpan,OK
8,Target minimum komentar,WARNING


Notebook 03 selesai dijalankan.
Silakan lanjut ke notebook 04 untuk audit dataset raw.
